### 08. Spatial and Administrative Harmonization for District-Level Climate & ESG Modelling

Construction of Spatial and Administrative Crosswalks for Integrating Regional Climate Indicators with District-Level Socioeconomic Data

**Data Inputs**

This notebook uses the following input datasets:

- Harmonized NUTS-2 climate datasets (Heatwave, Flood, Drought)
- Germany NUTS-2 administrative boundaries
- Germany NUTS-3 (Kreise) administrative boundaries
- Harmonized socioeconomic datasets (GDP per capita and Unemployment)

**Objective**

This notebook constructs the spatial and administrative harmonization framework required to integrate regional (NUTS-2) climate indicators with district-level (NUTS-3) socioeconomic data.

Specifically, it:

constructs a validated NUTS-2 → NUTS-3 spatial crosswalk;
investigates and resolves the documented 401-versus-400 district discrepancy;
develops an administrative crosswalk for the Eisenach–Wartburgkreis boundary reform;
harmonizes climate indicators from NUTS-2 to NUTS-3 using area-weighted interpolation; and
exports the crosswalks and harmonized climate datasets for construction of the project master dataset.


**Why do this**

The climate indicators are available only at the NUTS-2 regional level, whereas the socioeconomic datasets (GDP per capita and unemployment) are organized at the district (NUTS-3 / Kreise) level. To integrate these datasets into a common Climate & ESG Risk Intelligence framework, all variables must ultimately share the same spatial reference.

In addition, the climate and socioeconomic datasets are based on different administrative reference systems. The NUTS-3 boundary dataset contains 401 districts, whereas the harmonized INKAR socioeconomic datasets contain 400 districts because the former district Eisenach was incorporated into Wartburgkreis following the 2021 administrative reform. Furthermore, district identifiers and naming conventions differ between the climate and socioeconomic sources.

Therefore, before downscaling the climate indicators, this notebook first constructs validated spatial and administrative crosswalks to reconcile these differences. The validated crosswalks are then used to harmonize the NUTS-2 climate datasets to a consistent 400-district NUTS-3 framework, enabling direct integration with the harmonized socioeconomic datasets for subsequent Climate & ESG risk modelling.

**Workflow**

1. Load and validate NUTS-2 and NUTS-3 administrative boundaries.
2. Construct the NUTS-2 → NUTS-3 spatial crosswalk.
3. Resolve the 401-versus-400 district discrepancy.
4. Build the administrative crosswalk (AGS ↔ NUTS-3).
5. Downscale NUTS-2 climate indicators to NUTS-3 using the validated spatial crosswalk and area-weighted interpolation.
6. Validate spatial consistency and district coverage.
7. Export the crosswalks and harmonized climate datasets.

**Outputs**

- NUTS-2 → NUTS-3 Spatial Crosswalk
- Administrative Crosswalk (AGS ↔ NUTS-3)
- Harmonized NUTS-3 climate datasets (Heatwave, Flood, Drought)
- Spatial validation and quality assessment diagnostics

**Next Phase**

The harmonized climate datasets produced in this notebook will be integrated with the harmonized GDP per capita and unemployment datasets to construct the unified Climate & ESG Master Dataset. The master dataset will then serve as the input for Notebook 09, where a Feed-Forward Neural Network (FFNN) will be developed, trained, evaluated, and interpreted for district-level Climate & ESG Risk Intelligence.

# Part I — Build and Validate the Spatial Crosswalk (Area-Weight Table)

### Import Libraries

In [8]:
# ============================================================
# 0. Import libraries and settings
# ============================================================

import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path
from shapely.geometry import Polygon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

BASE_DIR = Path.cwd().parent          # Project root
RAW_CLIMATE_DIR = BASE_DIR / "data" / "raw" / "climate"

print("Project directory :", BASE_DIR)
print("Climate data path :", RAW_CLIMATE_DIR)
print("Path exists       :", RAW_CLIMATE_DIR.exists())

Project directory : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany
Climate data path : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate
Path exists       : True


##### Create Germany-only NUTS Boundary Files

In [11]:
# ============================================================
# 1. Create Germany-only NUTS boundary files
# ============================================================

# Input Europe-wide NUTS boundary files
nuts2_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_2.shp"
nuts3_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_3.shp"

# Output Germany-only NUTS boundary files
nuts2_de_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_de_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

# Check that input files exist
print("NUTS-2 Europe file exists:", nuts2_europe_path.exists())
print("NUTS-3 Europe file exists:", nuts3_europe_path.exists())

# Load Europe-wide NUTS files
nuts2 = gpd.read_file(nuts2_europe_path)
nuts3 = gpd.read_file(nuts3_europe_path)

# Filter Germany only
nuts2_de = nuts2[nuts2["CNTR_CODE"] == "DE"].copy()
nuts3_de = nuts3[nuts3["CNTR_CODE"] == "DE"].copy()

# Print diagnostics
print("\nEurope NUTS-2 shape:", nuts2.shape)
print("Germany NUTS-2 shape:", nuts2_de.shape)

print("\nEurope NUTS-3 shape:", nuts3.shape)
print("Germany NUTS-3 shape:", nuts3_de.shape)

print("\nNUTS-2 CRS:", nuts2_de.crs)
print("NUTS-3 CRS:", nuts3_de.crs)

# Save Germany-only boundary files
nuts2_de.to_file(nuts2_de_path)
nuts3_de.to_file(nuts3_de_path)

print("\nSaved Germany-only NUTS-2 to:", nuts2_de_path)
print("Saved Germany-only NUTS-3 to:", nuts3_de_path)

NUTS-2 Europe file exists: True
NUTS-3 Europe file exists: True

Europe NUTS-2 shape: (334, 18)
Germany NUTS-2 shape: (38, 18)

Europe NUTS-3 shape: (1514, 18)
Germany NUTS-3 shape: (401, 18)

NUTS-2 CRS: EPSG:4326
NUTS-3 CRS: EPSG:4326

Saved Germany-only NUTS-2 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts2_germany_2021.shp
Saved Germany-only NUTS-3 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts3_germany_2021.shp


##### Load Germany NUTS Administrative Boundaries

In [13]:
# ============================================================
# 2. Load Germany NUTS administrative boundaries
# ============================================================

nuts2_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

nuts2 = gpd.read_file(nuts2_path)
nuts3 = gpd.read_file(nuts3_path)

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

display(nuts2.head())
display(nuts3.head())

NUTS-2 regions: 38
NUTS-3 districts: 401


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE12,2,DE,Karlsruhe,Karlsruhe,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.45238 49.64864, 9.47674 49.6..."
1,DE14,2,DE,Tübingen,Tübingen,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((9.96528 48.59584, 9.9727 48.59462, 9..."
2,DE13,2,DE,Freiburg,Freiburg,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.00179 48.68692, 8.01441 48.6..."
3,DE21,2,DE,Oberbayern,Oberbayern,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((11.42642 49.06797, 11.44318 49..."
4,DEF0,2,DE,Schleswig-Holstein,Schleswig-Holstein,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.44242 55.05062, 8.46241 55.0..."


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE933,3,DE,Harburg,Harburg,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.78358 53.5, 9.78321 53.49212..."
1,DE934,3,DE,Lüchow-Dannenberg,Lüchow-Dannenberg,4,3,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((10.97163 53.1981, 11.01166 53.18148,..."
2,DE935,3,DE,"Lüneburg, Landkreis","Lüneburg, Landkreis",4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((10.55863 53.36796, 10.57365 53..."
3,DEA5A,3,DE,Siegen-Wittgenstein,Siegen-Wittgenstein,2,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.49703 51.13709, 8.50502 51.13271, ..."
4,DEA5B,3,DE,Soest,Soest,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.33538 51.72791, 8.34204 51.72319, ..."


##### Inspect Boundary Attributes

In [15]:
# ============================================================
# 3. Inspect boundary attributes
# ============================================================

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

print("\nCoordinate Reference System (CRS)")
print("NUTS-2:", nuts2.crs)
print("NUTS-3:", nuts3.crs)

print("\nKey attributes")
print("NUTS-2:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])
print("NUTS-3:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])

print("\nCountry codes")
print("NUTS-2:", nuts2["CNTR_CODE"].unique())
print("NUTS-3:", nuts3["CNTR_CODE"].unique())

NUTS-2 regions: 38
NUTS-3 districts: 401

Coordinate Reference System (CRS)
NUTS-2: EPSG:4326
NUTS-3: EPSG:4326

Key attributes
NUTS-2: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']
NUTS-3: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']

Country codes
NUTS-2: ['DE']
NUTS-3: ['DE']


## Build the Spatial Crosswalk (NUTS-2 → NUTS-3 Area Weights)

This step constructs the **Spatial Crosswalk (NUTS-2 → NUTS-3)** by calculating area-weight interpolation factors between Germany's NUTS-2 regions and NUTS-3 districts. The resulting area-weight table provides the spatial translation layer required to harmonize climate indicators from the NUTS-2 regional level to the NUTS-3 district level in a transparent and reproducible manner.

The resulting spatial crosswalk serves as the foundation for harmonizing all climate indicators that are currently available only at the NUTS-2 regional level. By calculating the proportion of each NUTS-3 district that overlaps with a NUTS-2 region, we derive area-weighting factors that are subsequently used to disaggregate regional climate indicators to the district level.

### Methodological Justification

Area-weight interpolation was selected because the climate indicators used in this project are available as aggregated regional statistics at the NUTS-2 level, while the target modelling framework operates at the finer NUTS-3 district level. In the absence of higher-resolution climate observations, the spatial crosswalk provides a transparent, reproducible, and computationally efficient approach for spatial harmonization.

### Methodological Assumptions

The harmonization assumes that each climate indicator represents a regional average that is spatially homogeneous within each NUTS-2 region. Consequently, the climate value assigned to a NUTS-3 district is estimated using the proportional area of overlap between the district and its parent NUTS-2 region.

The resulting area-weight table is generated once and subsequently reused for all climate indicators, ensuring methodological consistency across heatwave, flood, and drought preprocessing workflows.

In [55]:
# ============================================================
# Construct the Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ============================================================

# Use a projected CRS for accurate area calculations.
# EPSG:3035 is commonly used for Europe-wide equal-area analysis.
nuts2_proj = nuts2.to_crs(epsg=3035)
nuts3_proj = nuts3.to_crs(epsg=3035)

print("Projected CRS")
print("NUTS-2:", nuts2_proj.crs)
print("NUTS-3:", nuts3_proj.crs)

# Keep only required columns before overlay
nuts2_small = nuts2_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts2_id",
        "NUTS_NAME": "nuts2_name"
    }
)

nuts3_small = nuts3_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts3_id",
        "NUTS_NAME": "nuts3_name"
    }
)

# Calculate original NUTS-3 district area
nuts3_area = nuts3_small.copy()
nuts3_area["nuts3_area_m2"] = nuts3_area.geometry.area
nuts3_area = nuts3_area[["nuts3_id", "nuts3_area_m2"]]

# Compute spatial intersections required to construct the Spatial Crosswalk
intersections = gpd.overlay(
    nuts3_small,
    nuts2_small,
    how="intersection",
    keep_geom_type=False
)

# Calculate overlap area
intersections["overlap_area_m2"] = intersections.geometry.area

# Attach original NUTS-3 area and calculate area weight
area_weights = intersections.merge(nuts3_area, on="nuts3_id", how="left")
area_weights["area_weight"] = (
    area_weights["overlap_area_m2"] / area_weights["nuts3_area_m2"]
)

# Retain the final Spatial Crosswalk columns
area_weights = area_weights[
    [
        "nuts2_id",
        "nuts2_name",
        "nuts3_id",
        "nuts3_name",
        "overlap_area_m2",
        "nuts3_area_m2",
        "area_weight"
    ]
].copy()

# Sort the Spatial Crosswalk for reproducibility
area_weights = area_weights.sort_values(["nuts2_id", "nuts3_id"]).reset_index(drop=True)

print("Spatial Crosswalk (NUTS-2 → NUTS-3) created successfully.")
print("Spatial Crosswalk dimensions (rows, columns):", area_weights.shape)
print("Preview of the validated Spatial Crosswalk:")

display(area_weights.head())

Projected CRS
NUTS-2: EPSG:3035
NUTS-3: EPSG:3035
Spatial Crosswalk (NUTS-2 → NUTS-3) created successfully.
Spatial Crosswalk dimensions (rows, columns): (865, 7)
Preview of the validated Spatial Crosswalk:


,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
0,DE11,Stuttgart,DE111,"Stuttgart, Stadtkreis",2.101117e+08,2.101117e+08,1.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
3,DE11,Stuttgart,DE114,Göppingen,6.421750e+08,6.421750e+08,1.0
4,DE11,Stuttgart,DE115,Ludwigsburg,6.859905e+08,6.859905e+08,1.0


##### Initial Validation of the Spatial Crosswalk

In [56]:
# ============================================================
# 5a. Validate the initial Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    116
2    148
3    100
4     32
5      5
Name: count, dtype: int64


##### Inspect districts with multiple NUTS-2 intersections

In [57]:
# ============================================================
# 5b. Inspect districts with multiple NUTS-2 intersections
# ============================================================

multiple = (
    area_weights
    .groupby("nuts3_id")
    .filter(lambda x: len(x) > 1)
)

display(
    multiple.sort_values(["nuts3_id", "area_weight"])
)

,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
29,DE12,Karlsruhe,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
73,DE14,Tübingen,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
74,DE14,Tübingen,DE113,Esslingen,0.000000e+00,6.396529e+08,0.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
...,...,...,...,...,...,...,...
862,DEG0,Thüringen,DEG0M,Altenburger Land,5.698137e+08,5.698137e+08,1.0
400,DE73,Kassel,DEG0N,"Eisenach, Kreisfreie Stadt",0.000000e+00,1.046425e+08,0.0
863,DEG0,Thüringen,DEG0N,"Eisenach, Kreisfreie Stadt",1.046425e+08,1.046425e+08,1.0
401,DE73,Kassel,DEG0P,Wartburgkreis,0.000000e+00,1.305336e+09,0.0


**Interpretation**

Inspection of the multiple matches shows that the additional NUTS-2 assignments all have an area weight of 0.0, indicating that they arise solely from shared administrative boundaries rather than genuine spatial overlap. These records can therefore be safely removed without affecting the integrity of the resulting Spatial Crosswalk (NUTS-2 → NUTS-3).

##### Remove Zero-Area Boundary Artefacts

The initial spatial overlay identifies all geometric intersections between NUTS-2 and NUTS-3 polygons. Some of these correspond only to shared administrative boundaries and therefore have zero overlap area. These boundary artefacts are removed before constructing the final Spatial Crosswalk (NUTS-2 → NUTS-3) to ensure that each NUTS-3 district is associated only with its true parent NUTS-2 region.

In [58]:
area_weights = area_weights.query("area_weight > 0").copy()

##### Final Validation of the Spatial Crosswalk

In [59]:
# ============================================================
# 5c. Validate the Spatial Crosswalk
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    401
Name: count, dtype: int64


**Validation Outcome**

After removing zero-area boundary artefacts, each of Germany's 401 NUTS-3 districts is associated with exactly one NUTS-2 region. This confirms that the validated Spatial Crosswalk (NUTS-2 → NUTS-3) provides a reliable spatial translation layer for subsequent climate data harmonization.

#### Save the Validated Spatial Crosswalk

The validated **Spatial Crosswalk (NUTS-2 → NUTS-3)** is saved as a reusable project asset. It will be used by all subsequent climate harmonization workflows to transfer climate indicators from the NUTS-2 regional level to Germany's NUTS-3 districts using consistent area-weight interpolation.

In [60]:
# ==========================================================
# 7. Save the validated Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ==========================================================

OUTPUT_DIR = BASE_DIR / "data" / "processed" / "climate" / "spatial_lookup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

area_weights.to_csv(
    OUTPUT_DIR / "nuts2_to_nuts3_area_weights.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Spatial Crosswalk saved to: {OUTPUT_DIR}")
print(f"Validated Spatial Crosswalk districts: {len(area_weights):,}")

Spatial Crosswalk saved to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\spatial_lookup
Validated Spatial Crosswalk districts: 401


## Summary

In this notebook, a reusable Spatial Crosswalk (NUTS-2 → NUTS-3) was constructed and validated. The Spatial Crosswalk provides the area-weight interpolation factors required to harmonize climate indicators available at the NUTS-2 regional level to Germany's NUTS-3 districts.

The validated Spatial Crosswalk will be integrated into the climate harmonization workflows, enabling all climate indicators available at the NUTS-2 regional level to be consistently transferred to the NUTS-3 district level. The resulting harmonized climate datasets will provide the spatial foundation for integrating climate hazards with socioeconomic indicators and constructing the integrated Climate-ESG Risk Intelligence modelling dataset.

## Next Step

The validated Spatial Crosswalk (NUTS-2 → NUTS-3) will now be integrated into the heatwave, flood (Rx5day), and drought preprocessing workflows. Each climate indicator available at the NUTS-2 level will be transferred to the NUTS-3 district level using the validated area-weight interpolation approach, providing the spatial foundation for the integrated Climate–ESG Risk Intelligence modelling dataset.

# Data Integration Compatibility Check

### Integration Compatibility Check
### GISCO NUTS-3 vs. INKAR AGS Districts

Before harmonizing the climate datasets, verify that the GISCO NUTS-3 administrative units can be reliably integrated with the INKAR socio-economic datasets. This diagnostic identifies whether both data sources share a common district identifier or whether an intermediate Administrative Crosswalk (AGS → NUTS-3) is required.

##### Inspect the GISCO NUTS-3 Reference Table

In [61]:
# ================================================================
# Integration Check 1
# GISCO NUTS-3 Reference Table
# ================================================================

print("Rows:", len(area_weights))

print("\nColumns:")
print(area_weights.columns.tolist())

print("\nUnique NUTS-3 IDs:", area_weights["nuts3_id"].nunique())

display(
    area_weights[
        ["nuts3_id", "nuts3_name"]
    ].drop_duplicates().head(10)
)

Rows: 401

Columns:
['nuts2_id', 'nuts2_name', 'nuts3_id', 'nuts3_name', 'overlap_area_m2', 'nuts3_area_m2', 'area_weight']

Unique NUTS-3 IDs: 401


,nuts3_id,nuts3_name
0,DE111,"Stuttgart, Stadtkreis"
1,DE112,Böblingen
2,DE113,Esslingen
3,DE114,Göppingen
4,DE115,Ludwigsburg
5,DE116,Rems-Murr-Kreis
6,DE117,"Heilbronn, Stadtkreis"
7,DE118,"Heilbronn, Landkreis"
8,DE119,Hohenlohekreis
9,DE11A,Schwäbisch Hall


In [41]:
# ============================================================
# Raw socioeconomic datasets
# ============================================================

RAW_SOCIOECONOMIC_DIR = (
    BASE_DIR /
    "data" /
    "raw" /
    "socioeconomic"
)

##### Load one INKAR dataset

In [62]:
RAW_SOCIOECONOMIC_DIR = BASE_DIR / "data" / "raw" / "socioeconomic"

unemployment = pd.read_csv(
    RAW_SOCIOECONOMIC_DIR / "unemployment_inkar_1998_2023.csv"
)

print(unemployment.columns.tolist())
display(unemployment.head())
print(unemployment.shape)

['district_code', 'district_name', 'aggregation_level', 'year', 'unemployment_rate']


,district_code,district_name,aggregation_level,year,unemployment_rate
0,1001,"Flensburg, Stadt",Kreise,1998,14.60
1,1002,"Kiel, Landeshauptstadt",Kreise,1998,13.90
2,1003,"Lübeck, Hansestadt",Kreise,1998,13.41
3,1004,"Neumünster, Stadt",Kreise,1998,13.45
4,1051,Dithmarschen,Kreise,1998,10.70


(10400, 5)


##### Compare Identifiers / district names

In [63]:
# ============================================================
# Compare GISCO and INKAR identifiers
# ============================================================

print("INKAR columns:")
print(unemployment.columns.tolist())

print("\nUnique district codes:")
print(unemployment["district_code"].nunique())

display(
    unemployment[
        ["district_code", "district_name"]
    ].drop_duplicates().head(10)
)

INKAR columns:
['district_code', 'district_name', 'aggregation_level', 'year', 'unemployment_rate']

Unique district codes:
400


,district_code,district_name
0,1001,"Flensburg, Stadt"
1,1002,"Kiel, Landeshauptstadt"
2,1003,"Lübeck, Hansestadt"
3,1004,"Neumünster, Stadt"
4,1051,Dithmarschen
5,1053,Herzogtum Lauenburg
6,1054,Nordfriesland
7,1055,Ostholstein
8,1056,Pinneberg
9,1057,Plön


In [64]:
print("Minimum district code:", unemployment["district_code"].min())
print("Maximum district code:", unemployment["district_code"].max())

Minimum district code: 1001
Maximum district code: 16077


In [65]:
display(
    unemployment[
        ["district_code", "district_name"]
    ].drop_duplicates().sample(20, random_state=42)
)

,district_code,district_name
209,8327,Tuttlingen
280,9561,"Ansbach, Stadt"
33,3351,Celle
210,8335,Konstanz
93,5558,Coesfeld
84,5366,Euskirchen
329,12060,Barnim
94,5562,Recklinghausen
266,9377,Tirschenreuth
126,6437,Odenwaldkreis


##### Determine whether district_code is the official AGS/Kreisschlüssel

In [66]:
print("Rows:", len(unemployment))
print("Unique district codes:", unemployment["district_code"].nunique())

display(
    unemployment[
        ["district_code", "district_name"]
    ]
    .drop_duplicates()
    .sort_values("district_code")
    .tail(20)
)

Rows: 10400
Unique district codes: 400


,district_code,district_name
380,16053,"Jena, Stadt"
381,16054,"Suhl, Stadt"
382,16055,"Weimar, Stadt"
383,16061,Eichsfeld
384,16062,Nordhausen
385,16063,Wartburgkreis
386,16064,Unstrut-Hainich-Kreis
387,16065,Kyffhäuserkreis
388,16066,Schmalkalden-Meiningen
389,16067,Gotha


##### Compare one known district

For example:

GISCO

DE111
Stuttgart, Stadtkreis

In [67]:
unemployment[
    unemployment["district_name"]
    .str.contains("Stuttgart", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
178,8111,"Stuttgart, Stadtkreis",Kreise,1998,8.16
578,8111,"Stuttgart, Stadtkreis",Kreise,1999,7.32
978,8111,"Stuttgart, Stadtkreis",Kreise,2000,6.18
1378,8111,"Stuttgart, Stadtkreis",Kreise,2001,5.64
1778,8111,"Stuttgart, Stadtkreis",Kreise,2002,6.20
2178,8111,"Stuttgart, Stadtkreis",Kreise,2003,7.06
2578,8111,"Stuttgart, Stadtkreis",Kreise,2004,7.20
2978,8111,"Stuttgart, Stadtkreis",Kreise,2005,9.88
3378,8111,"Stuttgart, Stadtkreis",Kreise,2006,8.26
3778,8111,"Stuttgart, Stadtkreis",Kreise,2007,6.56


In [68]:
unemployment[
    unemployment["district_name"]
    .str.contains("Heilbronn", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
184,8121,"Heilbronn, Stadtkreis",Kreise,1998,9.81
185,8125,Heilbronn,Kreise,1998,6.74
584,8121,"Heilbronn, Stadtkreis",Kreise,1999,9.11
585,8125,Heilbronn,Kreise,1999,6.14
984,8121,"Heilbronn, Stadtkreis",Kreise,2000,7.70
985,8125,Heilbronn,Kreise,2000,4.82
1384,8121,"Heilbronn, Stadtkreis",Kreise,2001,6.97
1385,8125,Heilbronn,Kreise,2001,4.42
1784,8121,"Heilbronn, Stadtkreis",Kreise,2002,7.78
1785,8125,Heilbronn,Kreise,2002,4.86


In [69]:
unemployment[
    unemployment["district_name"]
    .str.contains("Pforzheim", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
199,8231,"Pforzheim, Stadtkreis",Kreise,1998,9.88
599,8231,"Pforzheim, Stadtkreis",Kreise,1999,9.11
999,8231,"Pforzheim, Stadtkreis",Kreise,2000,7.49
1399,8231,"Pforzheim, Stadtkreis",Kreise,2001,7.15
1799,8231,"Pforzheim, Stadtkreis",Kreise,2002,8.44
2199,8231,"Pforzheim, Stadtkreis",Kreise,2003,9.82
2599,8231,"Pforzheim, Stadtkreis",Kreise,2004,9.89
2999,8231,"Pforzheim, Stadtkreis",Kreise,2005,10.80
3399,8231,"Pforzheim, Stadtkreis",Kreise,2006,10.95
3799,8231,"Pforzheim, Stadtkreis",Kreise,2007,8.12


### Summary

In this integration compatibility assessment, the validated **Spatial Crosswalk (NUTS-2 → NUTS-3)** was compared with the INKAR socioeconomic datasets to verify that both data sources can be integrated into a common Climate–ESG modelling framework.

The inspection confirmed that the INKAR datasets contain the official German district identifier (`district_code`), corresponding to the official Amtlicher Gemeindeschlüssel (AGS/Kreisschlüssel), while the GISCO NUTS-3 reference table provides the corresponding NUTS-3 district identifiers and district names.

Manual validation using multiple districts, including Stuttgart, Heilbronn, and Pforzheim, confirmed that the district names are consistent between both datasets and that the AGS district codes provide a reliable bridge to the GISCO NUTS-3 administrative units.

This confirms that a permanent **Administrative Crosswalk (AGS → NUTS-3)** can be constructed and subsequently reused to integrate all INKAR socioeconomic datasets, including population, GDP per capita, unemployment, and future indicators, with the harmonized climate datasets.

A subsequent integration step will construct and validate the final **Administrative Crosswalk (AGS → NUTS-3)**, including verification of the remaining district count discrepancy between the GISCO and INKAR datasets.

The successful completion of this validation establishes the foundation for building the unified Climate–ESG master dataset.

### Resolve the 401 vs. 400 District Count Difference

This step compares the GISCO NUTS-3 district list with the INKAR district list to identify the source of the one-district difference between the two datasets.

In [70]:
# ============================================================
# Compare GISCO and INKAR district lists
# ============================================================

gisco_districts = (
    area_weights[["nuts3_id", "nuts3_name"]]
    .drop_duplicates()
    .sort_values("nuts3_id")
    .reset_index(drop=True)
)

inkar_districts = (
    unemployment[["district_code", "district_name"]]
    .drop_duplicates()
    .sort_values("district_code")
    .reset_index(drop=True)
)

print("GISCO NUTS-3 districts:", len(gisco_districts))
print("INKAR districts:", len(inkar_districts))

display(gisco_districts.head())
display(inkar_districts.head())

GISCO NUTS-3 districts: 401
INKAR districts: 400


,nuts3_id,nuts3_name
0,DE111,"Stuttgart, Stadtkreis"
1,DE112,Böblingen
2,DE113,Esslingen
3,DE114,Göppingen
4,DE115,Ludwigsburg


,district_code,district_name
0,1001,"Flensburg, Stadt"
1,1002,"Kiel, Landeshauptstadt"
2,1003,"Lübeck, Hansestadt"
3,1004,"Neumünster, Stadt"
4,1051,Dithmarschen


#### Compare district names

In [71]:
# ============================================================
# Compare district names
# ============================================================

gisco_names = set(gisco_districts["nuts3_name"])
inkar_names = set(inkar_districts["district_name"])

only_in_gisco = sorted(gisco_names - inkar_names)
only_in_inkar = sorted(inkar_names - gisco_names)

print("Names only in GISCO:", len(only_in_gisco))
print("Names only in INKAR:", len(only_in_inkar))

print("\nOnly in GISCO:")
print(only_in_gisco)

print("\nOnly in INKAR:")
print(only_in_inkar)

Names only in GISCO: 131
Names only in INKAR: 130

Only in GISCO:
['Amberg, Kreisfreie Stadt', 'Ansbach, Kreisfreie Stadt', 'Ansbach, Landkreis', 'Aschaffenburg, Kreisfreie Stadt', 'Aschaffenburg, Landkreis', 'Augsburg, Kreisfreie Stadt', 'Augsburg, Landkreis', 'Bamberg, Kreisfreie Stadt', 'Bamberg, Landkreis', 'Bayreuth, Kreisfreie Stadt', 'Bayreuth, Landkreis', 'Berlin', 'Bielefeld, Kreisfreie Stadt', 'Bochum, Kreisfreie Stadt', 'Bonn, Kreisfreie Stadt', 'Bottrop, Kreisfreie Stadt', 'Brandenburg an der Havel, Kreisfreie Stadt', 'Braunschweig, Kreisfreie Stadt', 'Bremen, Kreisfreie Stadt', 'Bremerhaven, Kreisfreie Stadt', 'Chemnitz, Kreisfreie Stadt', 'Coburg, Kreisfreie Stadt', 'Coburg, Landkreis', 'Cottbus, Kreisfreie Stadt', 'Darmstadt, Kreisfreie Stadt', 'Delmenhorst, Kreisfreie Stadt', 'Dessau-Roßlau, Kreisfreie Stadt', 'Dillingen a.d. Donau', 'Dortmund, Kreisfreie Stadt', 'Dresden, Kreisfreie Stadt', 'Duisburg, Kreisfreie Stadt', 'Düsseldorf, Kreisfreie Stadt', 'Eisenach, Kreisf

In [73]:
%whos

Variable                      Type            Data/Info
-------------------------------------------------------
BASE_DIR                      WindowsPath     d:\Purity\GitHub-Reposito<...>risk-intelligence-germany
OUTPUT_DIR                    WindowsPath     d:\Purity\GitHub-Reposito<...>ed\climate\spatial_lookup
PROCESSED_CLIMATE_DIR         WindowsPath     d:\Purity\GitHub-Reposito<...>ny\data\processed\climate
PROCESSED_SOCIOECONOMIC_DIR   WindowsPath     d:\Purity\GitHub-Reposito<...>a\processed\socioeconomic
Path                          type            <class 'pathlib.Path'>
Polygon                       type            <class 'shapely.geometry.polygon.Polygon'>
RAW_CLIMATE_DIR               WindowsPath     d:\Purity\GitHub-Reposito<...>-germany\data\raw\climate
RAW_SOCIOECONOMIC_DIR         WindowsPath     d:\Purity\GitHub-Reposito<...>ny\data\raw\socioeconomic
area_weights                  DataFrame       Shape: (401, 7)
district_counts               Series          Shape: (1,

In [74]:
intersections.columns.tolist()

['nuts3_id',
 'nuts3_name',
 'nuts2_id',
 'nuts2_name',
 'geometry',
 'overlap_area_m2']

#### Count GISCO districts by Bundesland

In [75]:
# ==========================================================
# District counts by Bundesland (GISCO NUTS-3)
# ==========================================================

gisco_state_counts = (
    area_weights[["nuts3_id"]]
    .drop_duplicates()
    .assign(state=lambda df: df["nuts3_id"].str[:3])   # e.g. DE1, DE2 ...
    .groupby("state")
    .size()
    .reset_index(name="gisco_districts")
    .sort_values("state")
)

display(gisco_state_counts)

print("Total GISCO districts:", gisco_state_counts["gisco_districts"].sum())

,state,gisco_districts
0,DE1,44
1,DE2,96
2,DE3,1
3,DE4,18
4,DE5,2
5,DE6,1
6,DE7,26
7,DE8,8
8,DE9,45
9,DEA,53


Total GISCO districts: 401


#### Count INKAR districts by Bundesland

In [76]:
# ==========================================================
# District counts by Bundesland (INKAR AGS)
# ==========================================================

inkar_state_counts = (
    unemployment[["district_code"]]
    .drop_duplicates()
    .assign(
        state=lambda df: (
            df["district_code"]
            .astype(str)
            .str.zfill(5)
            .str[:2]
        )
    )
    .groupby("state")
    .size()
    .reset_index(name="inkar_districts")
    .sort_values("state")
)

display(inkar_state_counts)

print("Total INKAR districts:", inkar_state_counts["inkar_districts"].sum())

,state,inkar_districts
0,01,15
1,02,1
2,03,45
3,04,2
4,05,53
5,06,26
6,07,36
7,08,44
8,09,96
9,10,6


Total INKAR districts: 400


#### Compare the two 

In [77]:
# ==========================================================
# Compare GISCO vs INKAR district counts by Bundesland
# ==========================================================

comparison = (
    gisco_state_counts.merge(
        inkar_state_counts,
        left_on="state",
        right_on="state",
        how="outer"
    )
    .fillna(0)
)

comparison["difference"] = (
    comparison["gisco_districts"] - comparison["inkar_districts"]
)

display(comparison)

,state,gisco_districts,inkar_districts,difference
0,01,0.0,15.0,-15.0
1,02,0.0,1.0,-1.0
2,03,0.0,45.0,-45.0
3,04,0.0,2.0,-2.0
4,05,0.0,53.0,-53.0
5,06,0.0,26.0,-26.0
6,07,0.0,36.0,-36.0
7,08,0.0,44.0,-44.0
8,09,0.0,96.0,-96.0
9,10,0.0,6.0,-6.0


### Standardize District Names

The apparent differences between the GISCO and INKAR district lists are expected because the two datasets use different naming conventions (e.g., "Landkreis", "kreisfreie Stadt", "Stadt").

To identify the true district discrepancy (401 vs. 400), both district name lists are standardized before comparison by removing administrative suffixes and formatting differences.

#### Create a standardization function

In [85]:
# ==========================================================
# Standardize district names
# ==========================================================

import re

def clean_district_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    # Normalize common German abbreviations before punctuation cleanup
    replacements = {
        "a.d.": "an der",
        "a. d.": "an der",
        "i.d.": "in der",
        "i. d.": "in der",
        "i.breisgau": "im breisgau",
        "i. breisgau": "im breisgau",
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    # Remove honorary / descriptive titles
    patterns = [
        r"\blandeshauptstadt\b",
        r"\bwissenschaftsstadt\b",
        r"\bklingenstadt\b",
        r"\bder fernuniversität\b",
        r"\bdocumenta[-\s]?stadt\b",
        r"\bdocumenta\b",
        r"\bhansestadt\b",
        r"\bkreisfreie stadt\b",
        r"\bstadtkreis\b",
        r"\blandkreis\b",
        r"\bstadt\b",
        r"\(de\)",
    ]

    for p in patterns:
        name = re.sub(p, "", name)

    # Remove punctuation after abbreviation normalization
    name = re.sub(r"[.,()\-]", " ", name)

    # Expand compressed German location expressions
    name = re.sub(r"ander", "an der ", name)
    name = re.sub(r"inder", "in der ", name)
    
    # Normalize Hamburg official name
    name = name.replace("freie und", "")

    # Collapse duplicate spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

#### Aply the Standardization Function

In [86]:
gisco_clean = (
    gisco_districts.assign(
        clean_name=gisco_districts["nuts3_name"].apply(clean_district_name)
    )
)

inkar_clean = (
    inkar_districts.assign(
        clean_name=inkar_districts["district_name"].apply(clean_district_name)
    )
)

gisco_names = set(gisco_clean["clean_name"])
inkar_names = set(inkar_clean["clean_name"])

only_gisco = sorted(gisco_names - inkar_names)
only_inkar = sorted(inkar_names - gisco_names)

print("Only in GISCO:", len(only_gisco))
print("Only in INKAR:", len(only_inkar))

print("\nGISCO only:")
print(only_gisco)

print("\nINKAR only:")
print(only_inkar)

Only in GISCO: 7
Only in INKAR: 6

GISCO only:
['dillingen an der donau', 'eisenach', 'neumarkt in der opf', 'neustadt an der aisch bad windsheim', 'neustadt an der waldnaab', 'pfaffenhofen an der ilm', 'weiden in der opf']

INKAR only:
['dillingen an derdonau', 'neumarkt in deropf', 'neustadt an deraisch bad windsheim', 'neustadt an derwaldnaab', 'pfaffenhofen an derilm', 'weiden in deropf']


#### Compare the two again

In [80]:
# ==========================================================
# Compare standardized names
# ==========================================================

gisco_names = set(gisco_clean["clean_name"])
inkar_names = set(inkar_clean["clean_name"])

only_gisco = sorted(gisco_names - inkar_names)
only_inkar = sorted(inkar_names - gisco_names)

print("Only in GISCO:", len(only_gisco))
print("Only in INKAR:", len(only_inkar))

print("\nGISCO only:")
print(only_gisco)

print("\nINKAR only:")
print(only_inkar)

Only in GISCO: 18
Only in INKAR: 19

GISCO only:
['darmstadt', 'dillingen a.d. donau', 'eisenach', 'friesland (de)', 'hagen', 'hamburg', 'kiel', 'lübeck', 'magdeburg', 'mühldorf a. inn', 'neumarkt i. d. opf.', 'neustadt a. d. aisch-bad windsheim', 'neustadt a. d. waldnaab', 'pfaffenhofen a. d. ilm', 'solingen', 'weiden i. d. opf', 'wiesbaden', 'wunsiedel i. fichtelgebirge']

INKAR only:
['darmstadt wissenschaftsstadt', 'dillingen a.d.donau', 'friesland', 'hagen der fernuniversität', 'hamburg freie und hansestadt', 'kassel documenta-', 'kiel landeshauptstadt', 'lübeck hansestadt', 'magdeburg landeshauptstadt', 'mühldorf a.inn', 'münchen landeshauptstadt', 'neumarkt i.d.opf.', 'neustadt a.d.aisch-bad windsheim', 'neustadt a.d.waldnaab', 'pfaffenhofen a.d.ilm', 'solingen klingenstadt', 'weiden i.d.opf.', 'wiesbaden landeshauptstadt', 'wunsiedel i.fichtelgebirge']


#### Inspect the remaining differences

In [81]:
# ==========================================================
# Remaining unmatched districts
# ==========================================================

display(
    gisco_clean[gisco_clean["clean_name"].isin(only_gisco)]
    .sort_values("clean_name")
)

display(
    inkar_clean[inkar_clean["clean_name"].isin(only_inkar)]
    .sort_values("clean_name")
)

,nuts3_id,nuts3_name,clean_name
162,DE711,"Darmstadt, Kreisfreie Stadt",darmstadt
132,DE277,Dillingen a.d. Donau,dillingen a.d. donau
399,DEG0N,"Eisenach, Kreisfreie Stadt",eisenach
233,DE94A,Friesland (DE),friesland (de)
284,DEA53,"Hagen, Kreisfreie Stadt",hagen
161,DE600,Hamburg,hamburg
364,DEF02,"Kiel, Kreisfreie Stadt",kiel
365,DEF03,"Lübeck, Kreisfreie Stadt",lübeck
351,DEE03,"Magdeburg, Kreisfreie Stadt",magdeburg
59,DE21G,Mühldorf a. Inn,mühldorf a. inn


,district_code,district_name,clean_name
116,6411,"Darmstadt, Wissenschaftsstadt",darmstadt wissenschaftsstadt
310,9773,Dillingen a.d.Donau,dillingen a.d.donau
53,3455,Friesland,friesland
106,5914,"Hagen, Stadt der FernUniversität",hagen der fernuniversität
15,2000,"Hamburg, Freie und Hansestadt",hamburg freie und hansestadt
135,6611,"Kassel, documenta-Stadt",kassel documenta-
1,1002,"Kiel, Landeshauptstadt",kiel landeshauptstadt
2,1003,"Lübeck, Hansestadt",lübeck hansestadt
366,15003,"Magdeburg, Landeshauptstadt",magdeburg landeshauptstadt
237,9183,Mühldorf a.Inn,mühldorf a.inn


#### Inspect remaining unmatched names exactly as stored

In [82]:
# ==========================================================
# Inspect remaining unmatched names exactly as stored
# ==========================================================

for name in sorted(only_gisco):
    print(repr(name))

print("\n" + "="*60 + "\n")

for name in sorted(only_inkar):
    print(repr(name))

'darmstadt'
'dillingen a.d. donau'
'eisenach'
'friesland (de)'
'hagen'
'hamburg'
'kiel'
'lübeck'
'magdeburg'
'mühldorf a. inn'
'neumarkt i. d. opf.'
'neustadt a. d. aisch-bad windsheim'
'neustadt a. d. waldnaab'
'pfaffenhofen a. d. ilm'
'solingen'
'weiden i. d. opf'
'wiesbaden'
'wunsiedel i. fichtelgebirge'


'darmstadt wissenschaftsstadt'
'dillingen a.d.donau'
'friesland'
'hagen der fernuniversität'
'hamburg freie und hansestadt'
'kassel documenta-'
'kiel landeshauptstadt'
'lübeck hansestadt'
'magdeburg landeshauptstadt'
'mühldorf a.inn'
'münchen landeshauptstadt'
'neumarkt i.d.opf.'
'neustadt a.d.aisch-bad windsheim'
'neustadt a.d.waldnaab'
'pfaffenhofen a.d.ilm'
'solingen klingenstadt'
'weiden i.d.opf.'
'wiesbaden landeshauptstadt'
'wunsiedel i.fichtelgebirge'


### Resolution of the 401 vs. 400 District Discrepancy

The compatibility assessment identified the source of the one-district difference between the GISCO NUTS-3 spatial dataset (401 districts) and the INKAR socio-economic datasets (400 districts).


| Dataset                 | District                   | Identifier                                                 |
| ----------------------- | -------------------------- | ---------------------------------------------------------- |
| **GISCO NUTS-3 (2021)** | Eisenach, Kreisfreie Stadt | **DEG0N**                                                  |
| **INKAR**               | Not listed separately      | Included within **Wartburgkreis (AGS 16063 / NUTS DEG0P)** |


#### Explanation

This discrepancy results from a genuine administrative boundary reform in the German state of Thüringen.

Until 30 June 2021, Eisenach existed as an independent Kreisfreie Stadt (urban district). On 1 July 2021, it was officially incorporated into the surrounding Wartburgkreis as part of the Thüringen district reorganization. Consequently, Eisenach ceased to exist as an independent district in the current German administrative structure.

The discrepancy arises because the two datasets reflect different administrative reference dates following the 2021 Thüringen district reform.

- GISCO NUTS 2021 still contains Eisenach (DEG0N) as a separate NUTS-3 region because the NUTS 2021 territorial classification was established before the July 2021 merger took effect. The corresponding Wartburgkreis is listed separately as DEG0P.
- INKAR, in contrast, reflects the post-merger German administrative structure and therefore reports 400 districts, with Eisenach incorporated into Wartburgkreis (AGS 16063 / NUTS DEG0P).


### Next Step

The spatial compatibility assessment confirms that the validated NUTS-2 → NUTS-3 Spatial Crosswalk provides a reliable basis for harmonizing the climate datasets.

The next phase of the project will temporarily shift to the socio-economic datasets, where the missing years will be imputed and the datasets finalized. Afterward, an Administrative Crosswalk (AGS ↔ NUTS-3) will be constructed to reconcile the documented Eisenach/Wartburgkreis administrative change and establish a consistent 400-district framework.

Once both the socio-economic datasets and the Administrative Crosswalk have been completed, the validated Spatial Crosswalk will be used to harmonize the heatwave, flood (Rx5day), and drought datasets from the NUTS-2 regional level to Germany's NUTS-3 district level using area-weighted interpolation. The harmonized climate datasets will then be integrated with the socio-economic indicators to construct the unified Climate–ESG master dataset.

#### Resolve the extra district (Eisenach)

Step 1 — Verify the Eisenach case

- Does the GISCO NUTS-3 layer contain Eisenach (DEG0N)?
- Does it also contain Wartburgkreis (DEG0P)?
- Does the INKAR dataset contain only Wartburgkreis (AGS 16063)?
- Does the current spatial crosswalk produce 401 mappings?

In [88]:
# Check available GeoDataFrame / DataFrame variable names
[name for name in globals() if "nuts3" in name.lower() or "crosswalk" in name.lower()]

['nuts3_europe_path',
 'nuts3_de_path',
 'nuts3',
 'nuts3_de',
 'nuts3_path',
 'nuts3_proj',
 'nuts3_small',
 'nuts3_area']

In [89]:
gdp_long = pd.read_csv("../data/processed/socioeconomic/gdp_per_capita_inkar_1990_2025.csv")

unemployment_long = pd.read_csv("../data/processed/socioeconomic/unemployment_inkar_1990_2025.csv")

In [90]:
# =============================================================================
# Load supporting datasets for administrative validation
# =============================================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

gdp_long = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "socioeconomic" / "gdp_per_capita_inkar_1990_2025.csv"
)

unemployment_long = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "socioeconomic" / "unemployment_inkar_1990_2025.csv"
)

print("GDP dataset loaded:", gdp_long.shape)
print("Unemployment dataset loaded:", unemployment_long.shape)

print("\nGDP columns:")
print(gdp_long.columns.tolist())

print("\nUnemployment columns:")
print(unemployment_long.columns.tolist())

GDP dataset loaded: (14400, 6)
Unemployment dataset loaded: (14400, 6)

GDP columns:
['district_code', 'district_name', 'aggregation_level', 'year', 'gdp_per_capita', 'imputation_flag']

Unemployment columns:
['district_code', 'district_name', 'aggregation_level', 'year', 'unemployment_rate', 'imputation_flag']


In [91]:
# =============================================================================
# Step 1 — Verify the Eisenach Case
# =============================================================================

print("=" * 80)
print("VERIFY EISENACH / WARTBURGKREIS ADMINISTRATIVE DISCREPANCY")
print("=" * 80)

# -----------------------------------------------------------------------------
# 1. Basic district counts
# -----------------------------------------------------------------------------

print("District counts:")
print(f"GISCO NUTS-3 districts        : {nuts3_de['NUTS_ID'].nunique()}")
print(f"INKAR GDP districts           : {gdp_long['district_code'].nunique()}")
print(f"INKAR Unemployment districts  : {unemployment_long['district_code'].nunique()}")

print()

# -----------------------------------------------------------------------------
# 2. Check Eisenach and Wartburgkreis in GISCO NUTS-3
# -----------------------------------------------------------------------------

eisenach_gisco = nuts3_de[
    nuts3_de["NUTS_ID"].astype(str).str.upper().eq("DEG0N")
].copy()

wartburg_gisco = nuts3_de[
    nuts3_de["NUTS_ID"].astype(str).str.upper().eq("DEG0P")
].copy()

print("GISCO NUTS-3 lookup:")
print(f"Eisenach present in GISCO?       {'Yes' if not eisenach_gisco.empty else 'No'}")
print(f"Wartburgkreis present in GISCO?  {'Yes' if not wartburg_gisco.empty else 'No'}")

gisco_display_cols = [
    col for col in ["NUTS_ID", "NAME_LATN", "NUTS_NAME", "LEVL_CODE"]
    if col in nuts3_de.columns
]

if not eisenach_gisco.empty:
    print("\nEisenach in GISCO:")
    display(eisenach_gisco[gisco_display_cols])

if not wartburg_gisco.empty:
    print("\nWartburgkreis in GISCO:")
    display(wartburg_gisco[gisco_display_cols])

print()

# -----------------------------------------------------------------------------
# 3. Check Eisenach and Wartburgkreis in INKAR socioeconomic datasets
# -----------------------------------------------------------------------------

gdp_districts = (
    gdp_long[["district_code", "district_name"]]
    .drop_duplicates()
    .copy()
)

unemployment_districts = (
    unemployment_long[["district_code", "district_name"]]
    .drop_duplicates()
    .copy()
)

# Ensure district codes are strings for comparison
gdp_districts["district_code"] = gdp_districts["district_code"].astype(str)
unemployment_districts["district_code"] = unemployment_districts["district_code"].astype(str)

# GDP lookup
gdp_eisenach = gdp_districts[
    gdp_districts["district_name"].str.contains("Eisenach", case=False, na=False)
]

gdp_wartburg = gdp_districts[
    (
        gdp_districts["district_name"].str.contains("Wartburg", case=False, na=False)
    )
    | (
        gdp_districts["district_code"].eq("16063")
    )
]

# Unemployment lookup
unemployment_eisenach = unemployment_districts[
    unemployment_districts["district_name"].str.contains("Eisenach", case=False, na=False)
]

unemployment_wartburg = unemployment_districts[
    (
        unemployment_districts["district_name"].str.contains("Wartburg", case=False, na=False)
    )
    | (
        unemployment_districts["district_code"].eq("16063")
    )
]

print("INKAR GDP lookup:")
print(f"Eisenach listed separately?      {'Yes' if not gdp_eisenach.empty else 'No'}")
print(f"Wartburgkreis / AGS 16063 found? {'Yes' if not gdp_wartburg.empty else 'No'}")

if not gdp_eisenach.empty:
    print("\nEisenach in GDP:")
    display(gdp_eisenach)

if not gdp_wartburg.empty:
    print("\nWartburgkreis in GDP:")
    display(gdp_wartburg)

print()

print("INKAR Unemployment lookup:")
print(f"Eisenach listed separately?      {'Yes' if not unemployment_eisenach.empty else 'No'}")
print(f"Wartburgkreis / AGS 16063 found? {'Yes' if not unemployment_wartburg.empty else 'No'}")

if not unemployment_eisenach.empty:
    print("\nEisenach in Unemployment:")
    display(unemployment_eisenach)

if not unemployment_wartburg.empty:
    print("\nWartburgkreis in Unemployment:")
    display(unemployment_wartburg)

print()

# -----------------------------------------------------------------------------
# 4. Summary conclusion
# -----------------------------------------------------------------------------

print("=" * 80)
print("SUMMARY")
print("=" * 80)

print("GISCO NUTS-3 includes Eisenach (DEG0N) as a separate district.")
print("GISCO NUTS-3 also includes Wartburgkreis (DEG0P) as a separate district.")
print("INKAR GDP and unemployment do not list Eisenach separately.")
print("INKAR uses the post-merger structure where Eisenach is included in Wartburgkreis.")

print()
print("✓ Eisenach / Wartburgkreis discrepancy verified.")

VERIFY EISENACH / WARTBURGKREIS ADMINISTRATIVE DISCREPANCY
District counts:
GISCO NUTS-3 districts        : 401
INKAR GDP districts           : 400
INKAR Unemployment districts  : 400

GISCO NUTS-3 lookup:
Eisenach present in GISCO?       Yes
Wartburgkreis present in GISCO?  Yes

Eisenach in GISCO:


,NUTS_ID,NAME_LATN,NUTS_NAME,LEVL_CODE
669,DEG0N,"Eisenach, Kreisfreie Stadt","Eisenach, Kreisfreie Stadt",3



Wartburgkreis in GISCO:


,NUTS_ID,NAME_LATN,NUTS_NAME,LEVL_CODE
769,DEG0P,Wartburgkreis,Wartburgkreis,3



INKAR GDP lookup:
Eisenach listed separately?      No
Wartburgkreis / AGS 16063 found? Yes

Wartburgkreis in GDP:


,district_code,district_name
13860,16063,Wartburgkreis



INKAR Unemployment lookup:
Eisenach listed separately?      No
Wartburgkreis / AGS 16063 found? Yes

Wartburgkreis in Unemployment:


,district_code,district_name
13860,16063,Wartburgkreis



SUMMARY
GISCO NUTS-3 includes Eisenach (DEG0N) as a separate district.
GISCO NUTS-3 also includes Wartburgkreis (DEG0P) as a separate district.
INKAR GDP and unemployment do not list Eisenach separately.
INKAR uses the post-merger structure where Eisenach is included in Wartburgkreis.

✓ Eisenach / Wartburgkreis discrepancy verified.


### Construct the Administrative Crosswalk

In [94]:
# =============================================================================
# Step 2 — Construct Complete Administrative Crosswalk
# =============================================================================

print("=" * 80)
print("CONSTRUCT ADMINISTRATIVE CROSSWALK")
print("=" * 80)

# Start from GISCO NUTS-3 districts
admin_crosswalk = nuts3_de.copy()

admin_crosswalk = admin_crosswalk[
    [col for col in ["NUTS_ID", "NAME_LATN", "NUTS_NAME", "LEVL_CODE"] if col in admin_crosswalk.columns]
].copy()

admin_crosswalk = admin_crosswalk.rename(
    columns={
        "NUTS_ID": "gisco_nuts3_id",
        "NAME_LATN": "gisco_name",
        "NUTS_NAME": "gisco_nuts_name",
        "LEVL_CODE": "nuts_level",
    }
)

# Default mapping: each GISCO district maps to itself
admin_crosswalk["target_nuts3_id"] = admin_crosswalk["gisco_nuts3_id"]
admin_crosswalk["target_name"] = admin_crosswalk["gisco_name"]
admin_crosswalk["target_ags"] = None
admin_crosswalk["mapping_status"] = "direct_match"
admin_crosswalk["requires_merge"] = False
admin_crosswalk["notes"] = "Direct GISCO NUTS-3 district mapping."

# Apply Eisenach → Wartburgkreis adjustment
eisenach_mask = admin_crosswalk["gisco_nuts3_id"].eq("DEG0N")

admin_crosswalk.loc[eisenach_mask, "target_nuts3_id"] = "DEG0P"
admin_crosswalk.loc[eisenach_mask, "target_name"] = "Wartburgkreis"
admin_crosswalk.loc[eisenach_mask, "target_ags"] = "16063"
admin_crosswalk.loc[eisenach_mask, "mapping_status"] = "merged_eisenach_wartburgkreis"
admin_crosswalk.loc[eisenach_mask, "requires_merge"] = True
admin_crosswalk.loc[eisenach_mask, "notes"] = (
    "Eisenach (DEG0N) existed separately in GISCO NUTS-3 2021 but was "
    "incorporated into Wartburgkreis on 1 July 2021. For this project, "
    "Eisenach is mapped to Wartburgkreis (DEG0P / AGS 16063) to align "
    "with the 400-district INKAR socioeconomic framework."
)

# Add AGS for Wartburgkreis target rows
admin_crosswalk.loc[
    admin_crosswalk["target_nuts3_id"].eq("DEG0P"),
    "target_ags"
] = "16063"

# Validate
print("Administrative crosswalk summary:")
print(f"GISCO NUTS-3 input districts     : {admin_crosswalk['gisco_nuts3_id'].nunique()}")
print(f"Target NUTS-3 output districts   : {admin_crosswalk['target_nuts3_id'].nunique()}")
print(f"Rows in administrative crosswalk : {len(admin_crosswalk)}")

print("\nMapping status:")
print(admin_crosswalk["mapping_status"].value_counts())

print("\nEisenach / Wartburgkreis mapping:")
display(
    admin_crosswalk[
        admin_crosswalk["gisco_nuts3_id"].isin(["DEG0N", "DEG0P"])
    ]
)

assert admin_crosswalk["gisco_nuts3_id"].nunique() == 401
assert admin_crosswalk["target_nuts3_id"].nunique() == 400
assert len(admin_crosswalk) == 401
assert admin_crosswalk["requires_merge"].sum() == 1

print("\n✓ Administrative crosswalk constructed and validated successfully.")

CONSTRUCT ADMINISTRATIVE CROSSWALK
Administrative crosswalk summary:
GISCO NUTS-3 input districts     : 401
Target NUTS-3 output districts   : 400
Rows in administrative crosswalk : 401

Mapping status:
mapping_status
direct_match                     400
merged_eisenach_wartburgkreis      1
Name: count, dtype: int64

Eisenach / Wartburgkreis mapping:


,gisco_nuts3_id,gisco_name,gisco_nuts_name,nuts_level,target_nuts3_id,target_name,target_ags,mapping_status,requires_merge,notes
669,DEG0N,"Eisenach, Kreisfreie Stadt","Eisenach, Kreisfreie Stadt",3,DEG0P,Wartburgkreis,16063,merged_eisenach_wartburgkreis,True,Eisenach (DEG0N) existed separately in GISCO N...
769,DEG0P,Wartburgkreis,Wartburgkreis,3,DEG0P,Wartburgkreis,16063,direct_match,False,Direct GISCO NUTS-3 district mapping.



✓ Administrative crosswalk constructed and validated successfully.


#### Export the Administrative Crosswalk

In [101]:
# =============================================================================
# Export Administrative Crosswalk
# =============================================================================

output_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crosswalks"
)

output_dir.mkdir(parents=True, exist_ok=True)

admin_crosswalk_file = output_dir / "administrative_crosswalk_gisco2021_to_project400.csv"

admin_crosswalk.to_csv(
    admin_crosswalk_file,
    index=False
)

print("=" * 80)
print("ADMINISTRATIVE CROSSWALK EXPORTED SUCCESSFULLY")
print("=" * 80)
print(admin_crosswalk_file)

ADMINISTRATIVE CROSSWALK EXPORTED SUCCESSFULLY
d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\crosswalks\administrative_crosswalk_gisco2021_to_project400.csv


#### Export Spatial Crosswalk

In [102]:
# =============================================================================
# Export Spatial Crosswalk
# =============================================================================

output_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crosswalks"
)

output_dir.mkdir(parents=True, exist_ok=True)

spatial_crosswalk_file = (
    output_dir
    / "spatial_crosswalk_nuts2_to_project400.csv"
)

final_project_spatial_crosswalk.to_csv(
    spatial_crosswalk_file,
    index=False
)

print("=" * 80)
print("SPATIAL CROSSWALK EXPORTED SUCCESSFULLY")
print("=" * 80)
print(spatial_crosswalk_file)

SPATIAL CROSSWALK EXPORTED SUCCESSFULLY
d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\crosswalks\spatial_crosswalk_nuts2_to_project400.csv


# Part II — Prepare the Climate Spatial Framework

#### Apply Administrative Crosswalk

- Load validated NUTS-2 → NUTS-3 Spatial Crosswalk
- Apply the Administrative Crosswalk
- Merge Eisenach (DEG0N) into Wartburgkreis (DEG0P)
- Validate the resulting 400-district framework
- Export Final Project Spatial Crosswalk


In [97]:
print("nuts2_de shape:", nuts2_de.shape)
print("nuts3_de shape:", nuts3_de.shape)

print("\nNUTS-2 unique IDs:", nuts2_de["NUTS_ID"].nunique())
print("NUTS-3 unique IDs:", nuts3_de["NUTS_ID"].nunique())

display(nuts2_de[["NUTS_ID", "NAME_LATN"]].head())
display(nuts3_de[["NUTS_ID", "NAME_LATN"]].head())

nuts2_de shape: (38, 18)
nuts3_de shape: (401, 18)

NUTS-2 unique IDs: 38
NUTS-3 unique IDs: 401


,NUTS_ID,NAME_LATN
0,DE12,Karlsruhe
11,DE14,Tübingen
23,DE13,Freiburg
24,DE21,Oberbayern
49,DEF0,Schleswig-Holstein


,NUTS_ID,NAME_LATN
1,DE933,Harburg
2,DE934,Lüchow-Dannenberg
6,DE935,"Lüneburg, Landkreis"
7,DEA5A,Siegen-Wittgenstein
8,DEA5B,Soest


In [99]:
# =============================================================================
# Apply Administrative Crosswalk and Build Final Project Spatial Crosswalk
# =============================================================================

print("=" * 80)
print("APPLY ADMINISTRATIVE CROSSWALK TO SPATIAL FRAMEWORK")
print("=" * 80)

# Project to equal-area CRS for area calculations
nuts2_proj = nuts2_de.to_crs(epsg=3035).copy()
nuts3_proj = nuts3_de.to_crs(epsg=3035).copy()

print("Projected CRS check:")
print("NUTS-2 CRS:", nuts2_proj.crs)
print("NUTS-3 CRS:", nuts3_proj.crs)

# Keep required columns
nuts2_proj = nuts2_proj[["NUTS_ID", "NAME_LATN", "geometry"]].rename(
    columns={"NUTS_ID": "nuts2_id", "NAME_LATN": "nuts2_name"}
)

nuts3_proj = nuts3_proj[["NUTS_ID", "NAME_LATN", "geometry"]].rename(
    columns={"NUTS_ID": "gisco_nuts3_id", "NAME_LATN": "gisco_nuts3_name"}
)

print("\nInput counts before overlay:")
print(f"NUTS-2 regions          : {nuts2_proj['nuts2_id'].nunique()}")
print(f"GISCO NUTS-3 districts  : {nuts3_proj['gisco_nuts3_id'].nunique()}")

# Compute NUTS-3 total areas
nuts3_proj["gisco_nuts3_area_km2"] = nuts3_proj.geometry.area / 1_000_000

# Spatial overlay
spatial_intersections = gpd.overlay(
    nuts3_proj,
    nuts2_proj,
    how="intersection",
    keep_geom_type=False
)

spatial_intersections["intersection_area_km2"] = (
    spatial_intersections.geometry.area / 1_000_000
)

# Remove only true zero-area artifacts
spatial_intersections = spatial_intersections[
    spatial_intersections["intersection_area_km2"] > 0
].copy()

print("\nOverlay output:")
print(f"Rows after overlay       : {len(spatial_intersections)}")
print(f"NUTS-2 represented       : {spatial_intersections['nuts2_id'].nunique()}")
print(f"NUTS-3 represented       : {spatial_intersections['gisco_nuts3_id'].nunique()}")

# Apply administrative crosswalk
spatial_admin = spatial_intersections.merge(
    admin_crosswalk[
        [
            "gisco_nuts3_id",
            "target_nuts3_id",
            "target_name",
            "target_ags",
            "mapping_status",
            "requires_merge",
        ]
    ],
    on="gisco_nuts3_id",
    how="left"
)

missing_admin = spatial_admin["target_nuts3_id"].isna().sum()
print(f"\nMissing administrative mappings: {missing_admin}")
assert missing_admin == 0

# Aggregate to target 400-district framework
final_project_spatial_crosswalk = (
    spatial_admin
    .groupby(
        [
            "nuts2_id",
            "nuts2_name",
            "target_nuts3_id",
        ],
        as_index=False
    )
    .agg(
        intersection_area_km2=("intersection_area_km2", "sum"),
        source_gisco_nuts3_count=("gisco_nuts3_id", "nunique"),
        requires_merge=("requires_merge", "max"),
    )
)

# Add target name and AGS after aggregation
target_lookup = (
    spatial_admin
    .sort_values(["target_nuts3_id", "requires_merge"], ascending=[True, False])
    .groupby("target_nuts3_id", as_index=False)
    .agg(
        target_name=("target_name", "first"),
        target_ags=("target_ags", "first"),
    )
)

final_project_spatial_crosswalk = final_project_spatial_crosswalk.merge(
    target_lookup,
    on="target_nuts3_id",
    how="left"
)

# Compute target NUTS-3 total area
target_area = (
    final_project_spatial_crosswalk
    .groupby("target_nuts3_id", as_index=False)["intersection_area_km2"]
    .sum()
    .rename(columns={"intersection_area_km2": "target_nuts3_area_km2"})
)

final_project_spatial_crosswalk = final_project_spatial_crosswalk.merge(
    target_area,
    on="target_nuts3_id",
    how="left"
)

final_project_spatial_crosswalk["area_weight"] = (
    final_project_spatial_crosswalk["intersection_area_km2"]
    / final_project_spatial_crosswalk["target_nuts3_area_km2"]
)

# Validation
weight_check = (
    final_project_spatial_crosswalk
    .groupby("target_nuts3_id")["area_weight"]
    .sum()
    .reset_index()
)

max_weight_error = (weight_check["area_weight"] - 1).abs().max()

print("\nFinal spatial crosswalk summary:")
print(f"NUTS-2 regions                    : {final_project_spatial_crosswalk['nuts2_id'].nunique()}")
print(f"Target NUTS-3 districts            : {final_project_spatial_crosswalk['target_nuts3_id'].nunique()}")
print(f"Rows in final spatial crosswalk    : {len(final_project_spatial_crosswalk)}")
print(f"Districts requiring merge          : {final_project_spatial_crosswalk['requires_merge'].sum()}")
print(f"Maximum area-weight error          : {max_weight_error:.10f}")

assert final_project_spatial_crosswalk["target_nuts3_id"].nunique() == 400
assert max_weight_error < 0.0001

print("\nEisenach/Wartburgkreis after administrative harmonization:")
display(
    final_project_spatial_crosswalk[
        final_project_spatial_crosswalk["target_nuts3_id"].eq("DEG0P")
    ]
)

print("\n✓ Final 400-district project spatial crosswalk created successfully.")

APPLY ADMINISTRATIVE CROSSWALK TO SPATIAL FRAMEWORK
Projected CRS check:
NUTS-2 CRS: EPSG:3035
NUTS-3 CRS: EPSG:3035

Input counts before overlay:
NUTS-2 regions          : 38
GISCO NUTS-3 districts  : 401

Overlay output:
Rows after overlay       : 401
NUTS-2 represented       : 38
NUTS-3 represented       : 401

Missing administrative mappings: 0

Final spatial crosswalk summary:
NUTS-2 regions                    : 38
Target NUTS-3 districts            : 400
Rows in final spatial crosswalk    : 400
Districts requiring merge          : 1
Maximum area-weight error          : 0.0000000000

Eisenach/Wartburgkreis after administrative harmonization:


,nuts2_id,nuts2_name,target_nuts3_id,intersection_area_km2,source_gisco_nuts3_count,requires_merge,target_name,target_ags,target_nuts3_area_km2,area_weight
399,DEG0,Thüringen,DEG0P,1409.978626,2,True,Wartburgkreis,16063,1409.978626,1.0



✓ Final 400-district project spatial crosswalk created successfully.


#### Export the final spatial crosswalk

In [103]:
# =============================================================================
# Export Spatial Overlay (Audit Trail)
# =============================================================================

overlay_file = (
    output_dir
    / "spatial_overlay_nuts2_gisco2021.csv"
)

spatial_admin.to_csv(
    overlay_file,
    index=False
)

print("=" * 80)
print("SPATIAL OVERLAY EXPORTED SUCCESSFULLY")
print("=" * 80)
print(overlay_file)

SPATIAL OVERLAY EXPORTED SUCCESSFULLY
d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\crosswalks\spatial_overlay_nuts2_gisco2021.csv


### Summary: Administrative and Spatial Crosswalk Construction

This section developed the administrative and spatial harmonization framework required to align all project datasets to the common Project400 district structure.

The administrative crosswalk converts the GISCO 2021 NUTS-3 district framework from 401 districts to the harmonized 400-district project framework. The key administrative adjustment is the merge of Eisenach into Wartburgkreis, represented by target NUTS-3 ID `DEG0P`.

The spatial crosswalk links the 38 German NUTS-2 regions to the harmonized 400 target NUTS-3 districts using area-weighted spatial overlay. This allows NUTS-2 climate indicators to be distributed consistently to the project’s NUTS-3 district framework.

### Validation Results

- 401 GISCO NUTS-3 districts were represented before harmonization.
- 400 target project districts were created after harmonization.
- 38 NUTS-2 regions were represented.
- Eisenach/Wartburgkreis was correctly merged.
- No administrative mappings were missing.
- Area weights summed to 1.0 for every target district.
- Maximum area-weight error was 0.0000000000.

### Exported Outputs

Three crosswalk outputs were exported to `data/processed/crosswalks/`:

- `administrative_crosswalk_gisco2021_to_project400.csv`
- `spatial_crosswalk_nuts2_to_project400.csv`
- `spatial_overlay_nuts2_gisco2021.csv`

These crosswalks now provide the validated administrative and spatial framework used to harmonize all climate and socioeconomic datasets before integration into the final master dataset.

## Harmonize Climate Datasets

#### 📝 Project400 Administrative Identifier Convention

The following convention is used consistently throughout the climate harmonization workflows in this notebook.

Throughout this project, harmonized datasets use the **Project400 GISCO NUTS-3 identifiers** (`target_nuts3_id`) from the administrative crosswalk as the canonical district identifiers.

These identifiers are used consistently across all harmonized climate and socioeconomic datasets, as well as the final Project400 master dataset.

- **Identifier standard:** GISCO NUTS-3 (`target_nuts3_id`)
- **AGS codes:** Not used in the harmonized datasets
- **Example identifiers:** `DE111`, `DE600`, `DEG0P`

This convention ensures a consistent administrative geography across all harmonized datasets and provides a common spatial reference for the final Project400 master dataset and subsequent modelling.

### Load Project400 Spatial Crosswalk

In [104]:
spatial_crosswalk = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crosswalks"
    / "spatial_crosswalk_nuts2_to_project400.csv"
)

print(spatial_crosswalk.shape)
display(spatial_crosswalk.head())

(400, 10)


,nuts2_id,nuts2_name,target_nuts3_id,intersection_area_km2,source_gisco_nuts3_count,requires_merge,target_name,target_ags,target_nuts3_area_km2,area_weight
0,DE11,Stuttgart,DE111,210.111728,1,False,"Stuttgart, Stadtkreis",NaN,210.111728,1.0
1,DE11,Stuttgart,DE112,616.706121,1,False,Böblingen,NaN,616.706121,1.0
2,DE11,Stuttgart,DE113,639.652917,1,False,Esslingen,NaN,639.652917,1.0
3,DE11,Stuttgart,DE114,642.175017,1,False,Göppingen,NaN,642.175017,1.0
4,DE11,Stuttgart,DE115,685.990467,1,False,Ludwigsburg,NaN,685.990467,1.0


**Harmonization steps**

    1. Load dataset
    2. Validate
    3. Merge with spatial crosswalk
    4. Apply area weights
    5. Aggregate to Project400
    6. Validate
    7. Export

### Harmonize Heatwave Dataset

In [116]:
# =============================================================================
# Harmonize Heatwave Dataset from NUTS-2 to Project400
# =============================================================================

import xarray as xr

# -----------------------------------------------------------------------------
# 1. Define input and output paths
# -----------------------------------------------------------------------------

heatwave_file = PROJECT_ROOT / "data" / "raw" / "climate" / "germany" / "heatwave_days_germany_nuts2_1990_2025.nc"
heatwave_output_file = PROJECT_ROOT / "data" / "processed" / "climate" / "heatwave_days_project400_1990_2025.csv"

heatwave_output_file.parent.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Prepare administrative crosswalk
# -----------------------------------------------------------------------------

admin_crosswalk = spatial_crosswalk.copy()

admin_crosswalk = admin_crosswalk.rename(
    columns={
        "target_nuts3_id": "district_code",
        "target_name": "district_name"
    }
)

admin_crosswalk["district_code"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str.strip()
)

admin_crosswalk["nuts2_id"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str[:4]
)

# -----------------------------------------------------------------------------
# 3. Load and reshape heatwave NetCDF dataset
# -----------------------------------------------------------------------------

heatwave_ds = xr.open_dataset(heatwave_file)
heatwave_variable = "climatological_heatwave_days"

heatwave_long = (
    heatwave_ds[heatwave_variable]
    .to_dataframe()
    .reset_index()
    .assign(year=lambda df: df["time"].dt.year)
    .rename(columns={"nuts": "nuts2_id", heatwave_variable: "heatwave_days"})
    [["nuts2_id", "year", "heatwave_days"]]
    .sort_values(["nuts2_id", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 4. Validate NUTS-2 input structure
# -----------------------------------------------------------------------------

expected_nuts2 = admin_crosswalk["nuts2_id"].nunique()
expected_years = 36

assert heatwave_long["nuts2_id"].nunique() == expected_nuts2
assert heatwave_long["year"].nunique() == expected_years
assert len(heatwave_long) == expected_nuts2 * expected_years
assert heatwave_long[["nuts2_id", "year"]].duplicated().sum() == 0

# -----------------------------------------------------------------------------
# 5. Expand NUTS-2 heatwave data to Project400 districts
# -----------------------------------------------------------------------------

heatwave_expanded = heatwave_long.merge(
    admin_crosswalk,
    on="nuts2_id",
    how="left",
    validate="many_to_many"
)

assert heatwave_expanded["district_code"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 6. Apply area weighting
# -----------------------------------------------------------------------------

heatwave_expanded["weighted_heatwave_days"] = (
    heatwave_expanded["heatwave_days"] * heatwave_expanded["area_weight"]
)

# -----------------------------------------------------------------------------
# 7. Aggregate to Project400 district-year level
# -----------------------------------------------------------------------------

heatwave_project400 = (
    heatwave_expanded
    .groupby(
        ["district_code", "district_name", "year"],
        as_index=False
    )
    .agg(
        heatwave_days=("weighted_heatwave_days", "sum")
    )
)

heatwave_project400["hazard"] = "heatwave"

heatwave_project400 = (
    heatwave_project400[
        ["district_code", "district_name", "year", "hazard", "heatwave_days"]
    ]
    .sort_values(["district_code", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 8. Validate Project400 output
# -----------------------------------------------------------------------------

assert heatwave_project400["district_code"].nunique() == 400
assert heatwave_project400["year"].nunique() == expected_years
assert len(heatwave_project400) == 400 * expected_years
assert heatwave_project400[["district_code", "year"]].duplicated().sum() == 0
assert heatwave_project400["heatwave_days"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 9. Export harmonized dataset
# -----------------------------------------------------------------------------

heatwave_project400.to_csv(heatwave_output_file, index=False)

# -----------------------------------------------------------------------------
# 10. Summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("HEATWAVE DATASET HARMONIZED SUCCESSFULLY")
print("=" * 80)
print("Input shape              :", heatwave_long.shape)
print("Expanded shape           :", heatwave_expanded.shape)
print("Output shape             :", heatwave_project400.shape)
print("NUTS-2 regions           :", heatwave_long["nuts2_id"].nunique())
print("Project400 districts     :", heatwave_project400["district_code"].nunique())
print("Years                    :", heatwave_project400["year"].min(), "-", heatwave_project400["year"].max())
print("Missing values           :", heatwave_project400["heatwave_days"].isna().sum())
print("District-year duplicates :", heatwave_project400[["district_code", "year"]].duplicated().sum())
print("Output file              :", heatwave_output_file)

display(heatwave_project400.head())
display(heatwave_project400.tail())

HEATWAVE DATASET HARMONIZED SUCCESSFULLY
Input shape              : (1368, 3)
Expanded shape           : (14400, 13)
Output shape             : (14400, 5)
NUTS-2 regions           : 38
Project400 districts     : 400
Years                    : 1990 - 2025
Missing values           : 0
District-year duplicates : 0
Output file              : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\heatwave_days_project400_1990_2025.csv


,district_code,district_name,year,hazard,heatwave_days
0,DE111,"Stuttgart, Stadtkreis",1990,heatwave,0.0
1,DE111,"Stuttgart, Stadtkreis",1991,heatwave,0.0
2,DE111,"Stuttgart, Stadtkreis",1992,heatwave,0.0
3,DE111,"Stuttgart, Stadtkreis",1993,heatwave,0.0
4,DE111,"Stuttgart, Stadtkreis",1994,heatwave,0.0


,district_code,district_name,year,hazard,heatwave_days
14395,DEG0P,Wartburgkreis,2021,heatwave,0.000000
14396,DEG0P,Wartburgkreis,2022,heatwave,0.000000
14397,DEG0P,Wartburgkreis,2023,heatwave,0.000000
14398,DEG0P,Wartburgkreis,2024,heatwave,0.000000
14399,DEG0P,Wartburgkreis,2025,heatwave,1.794337


### Harmonize Flood Dataset

In [119]:
# =============================================================================
# Harmonize Flood Dataset from NUTS-2 to Project400
# =============================================================================

import xarray as xr

# -----------------------------------------------------------------------------
# 1. Define input and output paths
# -----------------------------------------------------------------------------

flood_file = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "climate"
    / "germany"
    / "rx5day_germany_nuts2_1990_2025.nc"
)

flood_output_file = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "climate"
    / "rx5day_project400_1990_2025.csv"
)

flood_output_file.parent.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Prepare administrative crosswalk
# -----------------------------------------------------------------------------

admin_crosswalk = spatial_crosswalk.copy()

admin_crosswalk = admin_crosswalk.rename(
    columns={
        "target_nuts3_id": "district_code",
        "target_name": "district_name"
    }
)

admin_crosswalk["district_code"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str.strip()
)

admin_crosswalk["nuts2_id"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str[:4]
)

# -----------------------------------------------------------------------------
# 3. Load and reshape flood NetCDF dataset
# -----------------------------------------------------------------------------

flood_ds = xr.open_dataset(flood_file)

flood_variable = "maximum_five_day_precipitation"

flood_long = (
    flood_ds[flood_variable]
    .to_dataframe()
    .reset_index()
    .assign(year=lambda df: df["time"].dt.year)
    .rename(
        columns={
            "nuts": "nuts2_id",
            flood_variable: "rx5day"
        }
    )
    [["nuts2_id", "year", "rx5day"]]
    .sort_values(["nuts2_id", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 4. Validate NUTS-2 input structure
# -----------------------------------------------------------------------------

expected_nuts2 = admin_crosswalk["nuts2_id"].nunique()
expected_years = 36

assert flood_long["nuts2_id"].nunique() == expected_nuts2
assert flood_long["year"].nunique() == expected_years
assert len(flood_long) == expected_nuts2 * expected_years
assert flood_long[["nuts2_id", "year"]].duplicated().sum() == 0

# -----------------------------------------------------------------------------
# 5. Expand NUTS-2 flood data to Project400 districts
# -----------------------------------------------------------------------------

flood_expanded = flood_long.merge(
    admin_crosswalk,
    on="nuts2_id",
    how="left",
    validate="many_to_many"
)

assert flood_expanded["district_code"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 6. Apply area weighting
# -----------------------------------------------------------------------------

flood_expanded["weighted_rx5day"] = (
    flood_expanded["rx5day"]
    * flood_expanded["area_weight"]
)

# -----------------------------------------------------------------------------
# 7. Aggregate to Project400 district-year level
# -----------------------------------------------------------------------------

flood_project400 = (
    flood_expanded
    .groupby(
        ["district_code", "district_name", "year"],
        as_index=False
    )
    .agg(
        rx5day=("weighted_rx5day", "sum")
    )
)

flood_project400["hazard"] = "flood"

flood_project400 = (
    flood_project400[
        [
            "district_code",
            "district_name",
            "year",
            "hazard",
            "rx5day",
        ]
    ]
    .sort_values(["district_code", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 8. Validate Project400 output
# -----------------------------------------------------------------------------

assert flood_project400["district_code"].nunique() == 400
assert flood_project400["year"].nunique() == expected_years
assert len(flood_project400) == 400 * expected_years
assert flood_project400[["district_code", "year"]].duplicated().sum() == 0
assert flood_project400["rx5day"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 9. Export harmonized dataset
# -----------------------------------------------------------------------------

flood_project400.to_csv(flood_output_file, index=False)

# -----------------------------------------------------------------------------
# 10. Summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("FLOOD DATASET HARMONIZED SUCCESSFULLY")
print("=" * 80)
print("Input shape              :", flood_long.shape)
print("Expanded shape           :", flood_expanded.shape)
print("Output shape             :", flood_project400.shape)
print("NUTS-2 regions           :", flood_long["nuts2_id"].nunique())
print("Project400 districts     :", flood_project400["district_code"].nunique())
print("Years                    :", flood_project400["year"].min(), "-", flood_project400["year"].max())
print("Missing values           :", flood_project400["rx5day"].isna().sum())
print("District-year duplicates :", flood_project400[["district_code", "year"]].duplicated().sum())
print("Output file              :", flood_output_file)

display(flood_project400.head())
display(flood_project400.tail())

FLOOD DATASET HARMONIZED SUCCESSFULLY
Input shape              : (1368, 3)
Expanded shape           : (14400, 13)
Output shape             : (14400, 5)
NUTS-2 regions           : 38
Project400 districts     : 400
Years                    : 1990 - 2025
Missing values           : 0
District-year duplicates : 0
Output file              : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\rx5day_project400_1990_2025.csv


,district_code,district_name,year,hazard,rx5day
0,DE111,"Stuttgart, Stadtkreis",1990,flood,69.070652
1,DE111,"Stuttgart, Stadtkreis",1991,flood,71.949109
2,DE111,"Stuttgart, Stadtkreis",1992,flood,65.218925
3,DE111,"Stuttgart, Stadtkreis",1993,flood,84.942949
4,DE111,"Stuttgart, Stadtkreis",1994,flood,66.135985


,district_code,district_name,year,hazard,rx5day
14395,DEG0P,Wartburgkreis,2021,flood,66.051929
14396,DEG0P,Wartburgkreis,2022,flood,44.821277
14397,DEG0P,Wartburgkreis,2023,flood,56.612653
14398,DEG0P,Wartburgkreis,2024,flood,50.914976
14399,DEG0P,Wartburgkreis,2025,flood,49.664562


### Harmonize Drought Dataset

In [121]:
# =============================================================================
# Harmonize Drought Dataset from NUTS-2 to Project400
# =============================================================================

import xarray as xr

# -----------------------------------------------------------------------------
# 1. Define input and output paths
# -----------------------------------------------------------------------------

drought_file = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "climate"
    / "germany"
    / "drought_magnitude_germany_nuts2_1990_2025.nc"
)

drought_output_file = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "climate"
    / "drought_magnitude_project400_1990_2025.csv"
)

drought_output_file.parent.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Prepare administrative crosswalk
# -----------------------------------------------------------------------------

admin_crosswalk = spatial_crosswalk.copy()

admin_crosswalk = admin_crosswalk.rename(
    columns={
        "target_nuts3_id": "district_code",
        "target_name": "district_name"
    }
)

admin_crosswalk["district_code"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str.strip()
)

admin_crosswalk["nuts2_id"] = (
    admin_crosswalk["district_code"]
    .astype(str)
    .str[:4]
)

# -----------------------------------------------------------------------------
# 3. Load and reshape drought NetCDF dataset
# -----------------------------------------------------------------------------

drought_ds = xr.open_dataset(drought_file)

drought_variable = "meteorological_droughts_magnitude"

drought_long = (
    drought_ds[drought_variable]
    .to_dataframe()
    .reset_index()
    .assign(year=lambda df: df["time"].dt.year)
    .rename(
        columns={
            "nuts": "nuts2_id",
            drought_variable: "drought_magnitude"
        }
    )
    [["nuts2_id", "year", "drought_magnitude"]]
    .sort_values(["nuts2_id", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 4. Validate NUTS-2 input structure
# -----------------------------------------------------------------------------

expected_nuts2 = admin_crosswalk["nuts2_id"].nunique()
expected_years = 36

assert drought_long["nuts2_id"].nunique() == expected_nuts2
assert drought_long["year"].nunique() == expected_years
assert len(drought_long) == expected_nuts2 * expected_years
assert drought_long[["nuts2_id", "year"]].duplicated().sum() == 0

# -----------------------------------------------------------------------------
# 5. Expand NUTS-2 drought data to Project400 districts
# -----------------------------------------------------------------------------

drought_expanded = drought_long.merge(
    admin_crosswalk,
    on="nuts2_id",
    how="left",
    validate="many_to_many"
)

assert drought_expanded["district_code"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 6. Apply area weighting
# -----------------------------------------------------------------------------

drought_expanded["weighted_drought"] = (
    drought_expanded["drought_magnitude"]
    * drought_expanded["area_weight"]
)

# -----------------------------------------------------------------------------
# 7. Aggregate to Project400 district-year level
# -----------------------------------------------------------------------------

drought_project400 = (
    drought_expanded
    .groupby(
        ["district_code", "district_name", "year"],
        as_index=False
    )
    .agg(
        drought_magnitude=("weighted_drought", "sum")
    )
)

drought_project400["hazard"] = "drought"

drought_project400 = (
    drought_project400[
        [
            "district_code",
            "district_name",
            "year",
            "hazard",
            "drought_magnitude",
        ]
    ]
    .sort_values(["district_code", "year"])
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 8. Validate Project400 output
# -----------------------------------------------------------------------------

assert drought_project400["district_code"].nunique() == 400
assert drought_project400["year"].nunique() == expected_years
assert len(drought_project400) == 400 * expected_years
assert drought_project400[["district_code", "year"]].duplicated().sum() == 0
assert drought_project400["drought_magnitude"].isna().sum() == 0

# -----------------------------------------------------------------------------
# 9. Export harmonized dataset
# -----------------------------------------------------------------------------

drought_project400.to_csv(drought_output_file, index=False)

# -----------------------------------------------------------------------------
# 10. Summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("DROUGHT DATASET HARMONIZED SUCCESSFULLY")
print("=" * 80)
print("Input shape              :", drought_long.shape)
print("Expanded shape           :", drought_expanded.shape)
print("Output shape             :", drought_project400.shape)
print("NUTS-2 regions           :", drought_long["nuts2_id"].nunique())
print("Project400 districts     :", drought_project400["district_code"].nunique())
print("Years                    :", drought_project400["year"].min(), "-", drought_project400["year"].max())
print("Missing values           :", drought_project400["drought_magnitude"].isna().sum())
print("District-year duplicates :", drought_project400[["district_code", "year"]].duplicated().sum())
print("Output file              :", drought_output_file)

display(drought_project400.head())
display(drought_project400.tail())

DROUGHT DATASET HARMONIZED SUCCESSFULLY
Input shape              : (1368, 3)
Expanded shape           : (14400, 13)
Output shape             : (14400, 5)
NUTS-2 regions           : 38
Project400 districts     : 400
Years                    : 1990 - 2025
Missing values           : 0
District-year duplicates : 0
Output file              : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\drought_magnitude_project400_1990_2025.csv


,district_code,district_name,year,hazard,drought_magnitude
0,DE111,"Stuttgart, Stadtkreis",1990,drought,0.239709
1,DE111,"Stuttgart, Stadtkreis",1991,drought,8.796172
2,DE111,"Stuttgart, Stadtkreis",1992,drought,1.244216
3,DE111,"Stuttgart, Stadtkreis",1993,drought,2.221193
4,DE111,"Stuttgart, Stadtkreis",1994,drought,0.000000


,district_code,district_name,year,hazard,drought_magnitude
14395,DEG0P,Wartburgkreis,2021,drought,0.019809
14396,DEG0P,Wartburgkreis,2022,drought,3.250687
14397,DEG0P,Wartburgkreis,2023,drought,1.869792
14398,DEG0P,Wartburgkreis,2024,drought,0.000000
14399,DEG0P,Wartburgkreis,2025,drought,5.782439


# Part III — Build the Climate & ESG Master Dataset